In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 13})  # Adjust the font size as desired

# Impact of testing proportions

- The proportion of data available was varied for testing for every training cycle
- Proportions are [2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]

In [ ]:
# Results directory
results_dir = "/data1/GraphModellingExperiments/L10/TrainEval"
classes_type = "WithoutHands"
results_dir = os.path.join(results_dir, classes_type)
# Temporal information 
temporal_classes = ["12"]
temporal_classes.insert(0, "sc")

# Type of test
test_type = "unseenTest"


In [ ]:
# Final directory
data_dir = os.path.join(results_dir, "-".join(temporal_classes))
# Get all files inside
data_items = glob.glob(data_dir + os.sep + "*")

# Load all the csv files
df_dict = {}
for data_item in data_items:
    
    # Find all the csv files within
    csv_files_path = glob.glob(data_item + os.sep + "**" + os.sep + "*" + test_type + ".csv", recursive=True)
    df_dict[data_item.split(os.sep)[-1]] = {}
    for csv_file in csv_files_path:
        # Get the proportion
        proportion = int(csv_file.split(os.sep)[-2].split("-")[-1])
        df_dict[data_item.split(os.sep)[-1]][proportion] = pd.read_csv(csv_file, header="infer", index_col=0)


In [ ]:
# Plotting the results
metric_to_consider = "macro avg"
proportions = [2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
df_parsed_data = {}
for wo_config in df_dict.keys():
    df_parsed_data[wo_config] = []
    for proportion in proportions:
        df_parsed_data[wo_config].append(df_dict[wo_config][proportion].loc[metric_to_consider]["f1-score"])

# Plots
fig = plt.figure(figsize=(10, 8))
axs = fig.add_subplot([0, 0, 1, 1])

ordered_key_items = ["w30-o0", "w30-o10", "w30-o15", "w30-o25", "w15-o14", "w30-o29"]
for index, wo_config in enumerate(ordered_key_items):
    if wo_config not in df_parsed_data.keys():
        continue
    axs.plot(proportions, df_parsed_data[wo_config], label=wo_config, linewidth=3, color=plt.cm.tab10(index))
axs.legend()
axs.set_xticks(proportions)
axs.set_xlabel("Proportions of test data", fontsize=14)
axs.set_ylabel("Macro avg. f1-score", fontsize=14)
axs.set_title(f"Macro avg. f1 vs Proportions of test data (Temporal Class { '-'.join(temporal_classes)} )", fontsize=16)

# Impact of Overlap rate

In [ ]:
print(f"The class used for inference is {'-'.join(temporal_classes)}")

In [ ]:
# Get the average for a class
wo_averaged = {}
for wo_config in df_parsed_data.keys():
    wo_averaged[wo_config] = np.mean(df_parsed_data[wo_config][2:])


In [ ]:
# Bar charts for a class
# Plots
fig = plt.figure(figsize=(10, 8))
axs = fig.add_subplot([0, 0, 1, 1])

ordered_key_items = ["w30-o0", "w30-o10", "w30-o15", "w30-o25", "w15-o14", "w30-o29"]
for index, wo_config in enumerate(ordered_key_items):
    if wo_config not in wo_averaged.keys():
        continue
    
    # Compute the proportion
    overlap_rate = round(int(wo_config.split("-")[1][1:])/int(wo_config.split("-")[0][1:]), 2)
    
    axs.bar(wo_config + f"({overlap_rate})", wo_averaged[wo_config], width=0.5, color=plt.cm.tab10(index))

# Axis information
axs.set_xlabel("Overlap rate", fontsize=14)
axs.set_ylabel("F1-Score (Macro-Avg)", fontsize=14)
axs.set_title(f"Macro-Avg F1-Score vs Overlap Rate ({'-'.join(temporal_classes)})", fontsize=16)


# Compare between the two classes

In [ ]:
# Results directory
results_dir = "/data1/GraphModellingExperiments/L10/TrainEval"
classes_type = "WithoutHands"
results_dir = os.path.join(results_dir, classes_type)
# Temporal information 
temporal_classes_compared = ["11", "12", "11-12"]
temporal_classes_compared = ["sc-" + x for x in temporal_classes_compared]

# Type of test
test_type = "unseenTest"

In [ ]:
# Final directory
data_dirs = [os.path.join(results_dir, x) for x in temporal_classes_compared]

df_dict_class_combined = {}
for data_dir in data_dirs:
    # Get all files inside
    data_items = glob.glob(data_dir + os.sep + "*")
    
    # Load all the csv files
    df_dict = {}
    for data_item in data_items:
        
        # Find all the csv files within
        csv_files_path = glob.glob(data_item + os.sep + "**" + os.sep + "*" + test_type + ".csv", recursive=True)
        df_dict[data_item.split(os.sep)[-1]] = {}
        for csv_file in csv_files_path:
            # Get the proportion
            proportion = int(csv_file.split(os.sep)[-2].split("-")[-1])
            df_dict[data_item.split(os.sep)[-1]][proportion] = pd.read_csv(csv_file, header="infer", index_col=0)
            
    # Add by classes
    df_dict_class_combined[data_dir.split(os.sep)[-1]] = df_dict

In [ ]:
# Plotting the results
metric_to_consider = "macro avg"
proportions = [2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]

df_parsed_class_combined = {}
for class_type in df_dict_class_combined.keys():
    df_dict = df_dict_class_combined[class_type]
    df_parsed_data = {}
    for wo_config in df_dict.keys():
        df_parsed_data[wo_config] = []
        for proportion in proportions:
            df_parsed_data[wo_config].append(df_dict[wo_config][proportion].loc[metric_to_consider]["f1-score"])
            
    df_parsed_class_combined[class_type] = df_parsed_data

In [ ]:
# Get the average of the scores 
averaged_values = {}
for class_type in df_parsed_class_combined.keys():
    averaged_values[class_type] = {}
    for wo_config in df_parsed_class_combined[class_type].keys():
        averaged_values[class_type][wo_config] = np.mean(df_parsed_class_combined[class_type][wo_config][2:])


In [ ]:
# Plotting
# Plots
fig = plt.figure(figsize=(10, 8))
axs = fig.add_subplot([0, 0, 1, 1])

# Set the bar parameters
bar_width = 0.2

# Get the similar items across the two categories
for index, class_type in enumerate(averaged_values.keys()):
    if index == 0:
        items = list(averaged_values[class_type].items())
    else:
        if len(items) > len(list(averaged_values[class_type].items())):
            items = list(averaged_values[class_type].items())

ordered_key_items = ["w30-o0", "w30-o10", "w30-o15", "w30-o25", "w15-o14", "w30-o29"]
categories = []
classes_type = list(averaged_values.keys())
for index, wo_config in enumerate(ordered_key_items):
    
    availability = all([wo_config in averaged_values[x] for x in averaged_values.keys()])
    if not availability:
        continue
    
    # Compute the proportion
    overlap_rate = round(int(wo_config.split("-")[1][1:])/int(wo_config.split("-")[0][1:]), 2)
    categories.append(wo_config + f"({overlap_rate})")
    
    # Creating bars - For all the class types considered
    for i, class_type in enumerate(classes_type):
        values = averaged_values[class_type][wo_config] 
        axs.bar(index + i * bar_width, values, width=bar_width, color=plt.cm.tab10(i), edgecolor='black', label=class_type, linewidth=3,)
    
# Axis information
axs.set_xlabel("Overlap rate", fontsize=14)
axs.set_ylabel("F1-Score (Macro-Avg)", fontsize=14)
axs.set_title(f"Macro-Avg F1-Score vs Overlap Rate across classes ({'|'.join(classes_type)})", fontsize=16)
plt.xticks([r + bar_width/2 for r in range(len(categories))], categories)
axs.legend(classes_type)
    